In [1]:
#logistc regression
import pandas as pd
import time

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
recipes = pd.read_json("train.json")  
recipes.head()

,id,cuisine,ingredients
0,10259,greek,"[romaine lettuce, black olives, grape tomatoes..."
1,25693,southern_us,"[plain flour, ground pepper, salt, tomatoes, g..."
2,20130,filipino,"[eggs, pepper, salt, mayonaise, cooking oil, g..."
3,22213,indian,"[water, vegetable oil, wheat, salt]"
4,13162,indian,"[black pepper, shallots, cornflour, cayenne pe..."


In [3]:
print(recipes.shape)
print(recipes.columns)
print(recipes["cuisine"].nunique(), "cuisines")

(39774, 3)
Index(['id', 'cuisine', 'ingredients'], dtype='object')
20 cuisines


In [4]:
recipes["ingredients_text"] = recipes["ingredients"].apply(lambda xs: " ".join(xs).lower())
X = recipes["ingredients_text"]
y = recipes["cuisine"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [6]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),   # helps a lot
    min_df=2
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

print(X_train_vec.shape, X_test_vec.shape)

(27841, 31416) (11933, 31416)


In [8]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import time

# Train LinearSVC model
clf = LinearSVC(C=1, loss='hinge', penalty='l2', max_iter=10000)

# Measure training time
start_train = time.time()
clf.fit(X_train_vec, y_train)
end_train = time.time()
print(f"Training time: {end_train - start_train:.2f} seconds")

# Measure prediction time
start_pred = time.time()
y_pred = clf.predict(X_test_vec)
end_pred = time.time()
print(f"Prediction time: {end_pred - start_pred:.2f} seconds")

# Evaluate model performance
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Training time: 2.48 seconds
Prediction time: 0.02 seconds
Accuracy: 0.7805

Classification Report:
              precision    recall  f1-score   support

   brazilian       0.75      0.51      0.61       140
     british       0.53      0.35      0.42       241
cajun_creole       0.76      0.74      0.75       464
     chinese       0.78      0.87      0.83       802
    filipino       0.70      0.51      0.59       226
      french       0.60      0.61      0.61       794
       greek       0.75      0.66      0.70       352
      indian       0.84      0.91      0.88       901
       irish       0.67      0.42      0.52       200
     italian       0.80      0.89      0.84      2352
    jamaican       0.76      0.73      0.74       158
    japanese       0.82      0.72      0.77       427
      korean       0.79      0.75      0.77       249
     mexican       0.89      0.92      0.91      1932
    moroccan       0.77      0.75      0.76       246
     russian       0.62      0.41   

In [11]:
def predict_cuisine(ingredients):
    
    text = " ".join(ingredients).lower()
    
    vec = tfidf.transform([text])
    
    prediction = clf.predict(vec)
    
    return prediction[0]

In [12]:
predict_cuisine(["rice","salmon"])

'japanese'